# Environment setup

To reproduce this analysis:

1. Clone the repository
2. Create the conda environment:

conda env create -f environment.yml

3. Activate it:

conda activate saam_project

4. Open this notebook and run all cells from top to bottom.

# SAAM Project – Part I  
## 1. Data cleaning  

The datasets used in this notebook are cleaned versions of the original Datastream outputs.

The cleaning procedure follows the project guidelines:

- firms with completely missing rows in key datasets are removed  
- prices below 0.5 are treated as missing observations  
- missing prices at the beginning of the sample reflect firms not yet listed  
- missing prices at the end of the sample reflect delisting events  
- missing values are preserved in the return index dataset and handled later when constructing yearly investment sets  

## 2.1 Investment set construction  

This section constructs the investment universe used for the out-of-sample portfolio allocation.

Starting from the cleaned monthly return index data, we:

- compute monthly simple return series  
- define yearly investment sets based on data availability and carbon information  
- estimate rolling expected returns and covariance matrices using a 10-year historical window  

These quantities represent the necessary inputs for the portfolio optimization step described in Section 2.2.

### Step 1 — Load cleaned datasets

We load the cleaned monthly return index dataset, carbon emissions data, and static firm characteristics.

These datasets provide the inputs required to construct monthly return series and define the yearly investment universe. 
The cleaning of missing prices, missing internal values, delistings, and low prices has already been handled upstream in the cleaned files.

### Step 2 — Construct monthly simple returns

Monthly simple returns are computed from the return index series.

These return series will later be used to estimate:
- expected returns
- covariance matrices
- yearly investment sets based on data availability

In [37]:
import pandas as pd
import numpy as np

path = "../data/cleaned/"

ri_m = pd.read_csv(path + "RI_M_cleaned.csv", sep=";", na_values="N/A")
co2 = pd.read_csv(path + "CO2_S1_cleaned.csv", sep=";", na_values="N/A")
static = pd.read_csv(path + "STATIC_cleaned.csv", sep=";", na_values="N/A")

print("RI_M shape:", ri_m.shape)
print("CO2 shape:", co2.shape)
print("STATIC shape:", static.shape)

ri_m.head()

RI_M shape: (618, 316)
CO2 shape: (618, 29)
STATIC shape: (618, 4)


,NAME,ISIN,31/12/99,31/01/00,29/02/00,31/03/00,28/04/00,31/05/00,30/06/00,31/07/00,...,30/04/25,30/05/25,30/06/25,31/07/25,29/08/25,30/09/25,31/10/25,28/11/25,31/12/25,30/01/26
0,STRABAG SE,AT000000STR1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"430,27","430,83","479,46","477,35","465,06","461,5","396,65","452,87","481,49","526,88"
1,FLUGHAFEN WIEN,AT00000VIE62,"147,79","156,25","153,83","158,62","137,27","148,95","159,68","151,08",...,"2339,19","2371,03","2478,96","2417,14","2434,84","2425,66","2409,99","2533,4","2591,68","2587,67"
2,RAIFFEISEN BANK INTL.,AT0000606306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"113,07","129,28","128,7","123,36","140,73","146,05","158,13","171,78","190,56","215,11"
3,ERSTE GROUP BANK,AT0000652011,"102,94","94,91","97,74","100,5","96,13","97,86","102,06","102,43",...,"1237,75","1536,73","1621,45","1761,42","1818,07","1867,75","1979,03","2087,48","2308,86","2488,84"
4,TELEKOM AUSTRIA,AT0000720008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"343,5","359,25","383,12","360,02","370,54","361,64","360,3","350,11","357,49","360,93"


### Step 3 — Prepare the monthly return index dataset

The monthly return index dataset is stored in wide format.

The first columns identify firms (`NAME`, `ISIN`), while the remaining columns correspond to monthly observations already recorded as calendar dates.

Before computing returns, we:
- convert the date columns into proper monthly timestamps
- convert return index values into numeric format
- organize the dataset in matrix form, with firms on rows and months on columns

In [50]:
date_cols = ri_m.columns[2:].astype(str).str.strip()
dates = pd.Index([pd.to_datetime(c, dayfirst=True) for c in date_cols])

ri_m_wide = ri_m.copy()
ri_m_wide.columns = ["NAME", "ISIN"] + list(dates)

ri_m_wide = ri_m_wide.set_index(["ISIN", "NAME"])
ri_m_wide = ri_m_wide.sort_index(axis=1)

# Convert comma decimal values to numeric
ri_m_wide = ri_m_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("Prepared RI matrix:", ri_m_wide.shape)
ri_m_wide.iloc[:, :5]

Prepared RI matrix: (618, 314)


,,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00
ISIN,NAME,,,,,
AT000000STR1,STRABAG SE,NaN,NaN,NaN,NaN,NaN
AT00000VIE62,FLUGHAFEN WIEN,147.79,156.25,153.83,158.62,137.27
AT0000606306,RAIFFEISEN BANK INTL.,NaN,NaN,NaN,NaN,NaN
AT0000652011,ERSTE GROUP BANK,102.94,94.91,97.74,100.50,96.13
AT0000720008,TELEKOM AUSTRIA,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
SE0020050417,BOLIDEN ORD SHS,16.00,15.72,11.76,11.49,8.90
US0528001094,AUTOLIV,85.34,77.68,79.83,87.89,82.58
US70211M1099,PARTNER COMMS.ADR 1:1 DEAD - DELIST.22/09/23,159.82,113.50,105.00,102.69,66.01


### Step 4 — Compute monthly simple returns

Monthly simple returns are computed from the cleaned monthly return index series. Missing values are preserved during the calculation so that delistings and end-of-sample missing observations are handled consistently in the return series.

These monthly returns are the basis for constructing yearly investment sets and estimating rolling expected returns and covariance matrices.

In [40]:
returns_wide = ri_m_wide.pct_change(axis=1, fill_method=None)

print("Returns matrix shape:", returns_wide.shape)
returns_wide.iloc[:, :5]

Returns matrix shape: (618, 314)


,,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00
ISIN,NAME,,,,,
AT000000STR1,STRABAG SE,NaN,NaN,NaN,NaN,NaN
AT00000VIE62,FLUGHAFEN WIEN,NaN,0.057243,-0.015488,0.031138,-0.134598
AT0000606306,RAIFFEISEN BANK INTL.,NaN,NaN,NaN,NaN,NaN
AT0000652011,ERSTE GROUP BANK,NaN,-0.078007,0.029818,0.028238,-0.043483
AT0000720008,TELEKOM AUSTRIA,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
SE0020050417,BOLIDEN ORD SHS,NaN,-0.017500,-0.251908,-0.022959,-0.225413
US0528001094,AUTOLIV,NaN,-0.089759,0.027678,0.100965,-0.060416
US70211M1099,PARTNER COMMS.ADR 1:1 DEAD - DELIST.22/09/23,NaN,-0.289826,-0.074890,-0.022000,-0.357192


In [41]:
print("Total missing values in returns:", returns_wide.isna().sum().sum())
print("First return column entirely missing:", returns_wide.iloc[:, 0].isna().all())

Total missing values in returns: 8789
First return column entirely missing: True


### Step 5 — Prepare annual carbon data and define formation dates

We first reshape the annual Scope 1 carbon emissions dataset into matrix form, with firms on rows and calendar years on columns.

We then identify all December dates available in the monthly return dataset. These December dates correspond to the year-end formation dates at which the investment set is defined and the portfolio inputs are estimated.

In [42]:
co2_year_cols = co2.columns[2:]
co2_years = pd.to_numeric(co2_year_cols, errors="raise").astype(int)

co2_wide = co2.copy()
co2_wide.columns = ["NAME", "ISIN"] + list(co2_years)
co2_wide = co2_wide.set_index(["ISIN", "NAME"])

# Convert comma decimals to numeric
co2_wide = co2_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("Prepared CO2 matrix:", co2_wide.shape)

december_dates = [d for d in returns_wide.columns if d.month == 12]
years_available = [d.year for d in december_dates]

print("December years available:", years_available[:5], "...", years_available[-5:])

Prepared CO2 matrix: (618, 27)
December years available: [1999, 2000, 2001, 2002, 2003] ... [2021, 2022, 2023, 2024, 2025]


### Step 5.1 — Filter 1: sufficient return history

For each year-end formation date \(Y\), we consider the 10-year rolling window of monthly returns ending in December of year \(Y\).

Following the project instructions, a firm is retained only if it has a sufficient number of valid monthly return observations in that window. In line with the suggested rule, we require at least 36 valid monthly returns over the previous 120 months.

In [43]:
sufficient_history_by_year = {}
window_returns_by_year = {}
common_index_by_year = {}

for dec_date in december_dates:
    # Require a full 10-year rolling window (120 months)
    window_start = dec_date - pd.DateOffset(months=119)

    if window_start < returns_wide.columns[0]:
        continue

    Y = dec_date.year

    # Start the allocation exercise at end-2013
    if Y < 2013:
        continue

    # Last formation year = 2024 (portfolio used in 2025)
    if Y > 2024:
        continue

    # Rolling monthly return window up to end of year Y
    window_returns = returns_wide.loc[:, window_start:dec_date]

    # Firms available in both returns and CO2 datasets
    common_index = returns_wide.index.intersection(co2_wide.index)

    # Filter 1: sufficient return history
    valid_return_counts = window_returns.notna().sum(axis=1)
    sufficient_history = valid_return_counts >= 36

    window_returns_by_year[Y] = window_returns
    common_index_by_year[Y] = common_index
    sufficient_history_by_year[Y] = sufficient_history

print("Computed sufficient-history filter for", len(sufficient_history_by_year), "years.")

Computed sufficient-history filter for 12 years.


### Step 5.2 — Filter 2: stale prices exclusion

We then apply the stale-price filter suggested in the project instructions.

For each firm and each year \(Y\), we compute the proportion of months with zero return over the 10-year estimation window ending in December of year \(Y\). If this proportion exceeds 50%, the firm is treated as subject to stale prices and is excluded from the investment set.

In [44]:
stale_prices_ok_by_year = {}

for Y, window_returns in window_returns_by_year.items():
    # Proportion of zero monthly returns over the full 10-year window
    zero_return_ratio = (window_returns == 0).sum(axis=1) / 120

    # Filter 2: exclude firms with more than 50% zero returns
    stale_prices_ok = zero_return_ratio <= 0.50

    stale_prices_ok_by_year[Y] = stale_prices_ok

print("Computed stale-price filter for", len(stale_prices_ok_by_year), "years.")

Computed stale-price filter for 12 years.


### Step 5.3 — Filter 3: carbon data availability and final investment set

Finally, we exclude firms without Scope 1 carbon data available at the end of year \(Y\).

The final investment set for each year \(Y\) is obtained by combining:
- the sufficient return history filter,
- the stale-price filter,
- and the carbon availability filter.

This ensures that the same yearly investment universe can be used consistently in both parts of the project.

In [45]:
carbon_available_by_year = {}
investment_sets = {}

for Y in window_returns_by_year.keys():
    common_index = common_index_by_year[Y]

    # Filter 3: carbon data available at end of year Y
    if Y in co2_wide.columns:
        carbon_available = co2_wide[Y].notna()
    else:
        carbon_available = pd.Series(False, index=co2_wide.index)

    carbon_available_by_year[Y] = carbon_available

    eligible = common_index[
        sufficient_history_by_year[Y].loc[common_index] &
        stale_prices_ok_by_year[Y].loc[common_index] &
        carbon_available.loc[common_index]
    ]

    investment_sets[Y] = list(eligible)

investment_set_sizes = pd.Series({year: len(firms) for year, firms in investment_sets.items()})

print("Number of yearly investment sets:", len(investment_sets))
print("First investable year:", investment_set_sizes.index[0])
print("Size first set:", investment_set_sizes.iloc[0])
print("Last investable year:", investment_set_sizes.index[-1])
print("Size last set:", investment_set_sizes.iloc[-1])

investment_set_sizes

Number of yearly investment sets: 12
First investable year: 2013
Size first set: 477
Last investable year: 2024
Size last set: 613


2013    477
2014    495
2015    511
2016    518
2017    532
2018    551
2019    577
2020    597
2021    605
2022    613
2023    613
2024    613
dtype: int64

In [46]:
first_year = investment_set_sizes.index[0]
first_set = investment_sets[first_year]

print("First investable year:", first_year)
print("Number of firms:", len(first_set))
first_set[:10]

First investable year: 2013
Number of firms: 477


[('AT000000STR1', 'STRABAG SE'),
 ('AT0000606306', 'RAIFFEISEN BANK INTL.'),
 ('AT0000652011', 'ERSTE GROUP BANK'),
 ('AT0000720008', 'TELEKOM AUSTRIA'),
 ('AT0000743059', 'OMV'),
 ('AT0000746409', 'VERBUND'),
 ('BE0003470755', 'SOLVAY'),
 ('BE0003565737', 'KBC GROUP'),
 ('BE0003593044', 'COFINIMMO'),
 ('BE0003735496', 'ORANGE BELGIUM')]

### Step 6 — Compute rolling expected returns and covariance matrices

For each investable year \(Y\), we estimate the inputs required for the minimum-variance optimization.

Using the firms belonging to the investment set of year \(Y\), we compute:
- the vector of expected returns \(\hat{\mu}_Y\), as the average of monthly returns over the 10-year rolling window up to December of year \(Y\),
- the covariance matrix \(\Sigma_Y\), using the same rolling window.

These quantities will serve as inputs for the out-of-sample portfolio allocation step.

In [47]:
mu_by_year = {}
cov_by_year = {}
returns_window_by_year = {}

for Y, firms in investment_sets.items():
    # December date associated with year Y
    dec_date = pd.Timestamp(f"{Y}-12-31")
    
    # Rolling 10-year window = 120 months up to end of year Y
    window_start = dec_date - pd.DateOffset(months=119)
    window_returns = returns_wide.loc[firms, window_start:dec_date].copy()
    
    # Store returns window for later use in portfolio construction
    returns_window_by_year[Y] = window_returns
    
    # Expected returns: average over available monthly returns
    mu_y = window_returns.mean(axis=1)
    
    # Covariance matrix
    cov_y = window_returns.T.cov()
    
    mu_by_year[Y] = mu_y
    cov_by_year[Y] = cov_y

print("Number of expected return vectors:", len(mu_by_year))
print("Number of covariance matrices:", len(cov_by_year))

first_year = list(mu_by_year.keys())[0]
print("First year:", first_year)
print("Mu vector size:", mu_by_year[first_year].shape)
print("Cov matrix shape:", cov_by_year[first_year].shape)

Number of expected return vectors: 12
Number of covariance matrices: 12
First year: 2013
Mu vector size: (477,)
Cov matrix shape: (477, 477)


In [48]:
first_year = list(mu_by_year.keys())[0]

print("First investable year:", first_year)
print("\nExpected returns (first 10 firms):")
display(mu_by_year[first_year].head(10))

print("\nCovariance matrix (top-left 5x5 block):")
display(cov_by_year[first_year].iloc[:5, :5])

First investable year: 2013

Expected returns (first 10 firms):


ISIN          NAME                 
AT000000STR1  STRABAG SE              -0.000342
AT0000606306  RAIFFEISEN BANK INTL.    0.009210
AT0000652011  ERSTE GROUP BANK         0.012839
AT0000720008  TELEKOM AUSTRIA          0.002051
AT0000743059  OMV                      0.017438
AT0000746409  VERBUND                  0.011337
BE0003470755  SOLVAY                   0.011613
BE0003565737  KBC GROUP                0.015691
BE0003593044  COFINIMMO                0.004985
BE0003735496  ORANGE BELGIUM          -0.001966
dtype: float64


Covariance matrix (top-left 5x5 block):


,ISIN,AT000000STR1,AT0000606306,AT0000652011,AT0000720008,AT0000743059
,NAME,STRABAG SE,RAIFFEISEN BANK INTL.,ERSTE GROUP BANK,TELEKOM AUSTRIA,OMV
ISIN,NAME,,,,,
AT000000STR1,STRABAG SE,0.020084,0.015876,0.018473,0.007008,0.009323
AT0000606306,RAIFFEISEN BANK INTL.,0.015876,0.020598,0.017885,0.005767,0.008832
AT0000652011,ERSTE GROUP BANK,0.018473,0.017885,0.020441,0.005075,0.007458
AT0000720008,TELEKOM AUSTRIA,0.007008,0.005767,0.005075,0.007252,0.004296
AT0000743059,OMV,0.009323,0.008832,0.007458,0.004296,0.011011


In [52]:
for Y in cov_by_year:
    cov_y = cov_by_year[Y]
    assert cov_y.shape[0] == cov_y.shape[1], f"Covariance matrix not square in year {Y}"
    assert cov_y.index.equals(cov_y.columns), f"Covariance matrix labels mismatch in year {Y}"

print("All covariance matrices are square and properly labelled.")

All covariance matrices are square and properly labelled.


## 2.2 Minimum-variance portfolio allocation

In [62]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =========================================================
# SETTINGS
# =========================================================
# Notebook-friendly paths:
# run this from the notebooks folder, so ../data/cleaned exists
DATA_DIR = Path("../data/cleaned")
RI_FILE = DATA_DIR / "RI_M_cleaned.csv"
CO2_FILE = DATA_DIR / "CO2_S1_cleaned.csv"
RF_FILE = DATA_DIR / "RF.csv"

OUTPUT_DIR = Path("../results/outputs_minvar")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_FORMATION_YEAR = 2013   # first portfolio formed at end-2013
END_FORMATION_YEAR = 2024     # last portfolio formed at end-2024
MIN_OBS_WINDOW = 36           # at least 3 years of monthly returns in rolling 10y window
STALE_ZERO_THRESHOLD = 0.50   # exclude if more than 50% of returns are zero
RIDGE = 1e-8                  # numerical stabilization for covariance matrix
SOLVER_MAXITER = 2000
FTOL = 1e-12
CSV_SEP = ";"

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================================================
# HELPERS
# =========================================================
def to_numeric_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Convert object cells using comma decimal if present."""
    return df.apply(
        lambda col: pd.to_numeric(
            col.astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
    )


def parse_calendar_date_headers(columns: list[str]) -> list[pd.Timestamp]:
    """
    Convert string date headers like 31/12/1999 into pandas Timestamps.
    We parse one by one to avoid pandas format-inference warnings.
    """
    cols = pd.Index(columns).astype(str).str.strip()
    return [pd.to_datetime(c, dayfirst=True) for c in cols]


def parse_rf_dates(date_series: pd.Series) -> pd.DatetimeIndex:
    """
    Parse RF dates robustly:
    - if they look numeric, treat them as Excel serial dates
    - otherwise parse them as day-first calendar dates
    """
    raw = date_series.astype(str).str.strip()
    numeric = pd.to_numeric(raw, errors="coerce")

    if numeric.notna().all():
        return pd.DatetimeIndex(pd.to_datetime(numeric, unit="D", origin="1899-12-30"))

    return pd.DatetimeIndex(pd.to_datetime(raw, dayfirst=True, errors="coerce"))


def annualized_return(monthly_returns: pd.Series) -> float:
    r = monthly_returns.dropna()
    if len(r) == 0:
        return np.nan
    compounded = (1.0 + r).prod()
    return compounded ** (12.0 / len(r)) - 1.0


def annualized_volatility(monthly_returns: pd.Series) -> float:
    r = monthly_returns.dropna()
    if len(r) < 2:
        return np.nan
    return r.std(ddof=1) * np.sqrt(12.0)


def max_drawdown(cumulative_index: pd.Series) -> float:
    running_max = cumulative_index.cummax()
    drawdown = cumulative_index / running_max - 1.0
    return drawdown.min()


# =========================================================
# LOAD CLEANED CSV FILES
# =========================================================
print("Loading cleaned CSV files...")

ri_raw = pd.read_csv(RI_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])
co2_raw = pd.read_csv(CO2_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])
rf_raw = pd.read_csv(RF_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])

# =========================================================
# PREPARE MONTHLY RI PANEL
# =========================================================
print("Preparing monthly RI panel...")

# NOTE: in the new cleaned RI_M file, date columns are already calendar dates,
# not Excel serial numbers.
ri_date_cols = parse_calendar_date_headers(list(ri_raw.columns[2:]))

ri = ri_raw.copy()
ri.columns = ["NAME", "ISIN"] + ri_date_cols
ri = ri.set_index(["ISIN", "NAME"])
ri = to_numeric_panel(ri)

# NOTE:
# We do NOT mask RI < 0.5 here, because the cleaned CSV is assumed to
# already reflect your chosen data-cleaning rules, and explicit delisting
# zeros must be preserved.

# Official project output ends at Dec-2025
ri = ri.loc[:, ri.columns <= pd.Timestamp("2025-12-31")]

print("RI panel shape:", ri.shape)

# =========================================================
# CONSTRUCT MONTHLY RETURNS
# =========================================================
print("Constructing monthly simple returns...")

# IMPORTANT:
# fill_method=None preserves missing values and keeps the delisting treatment
# already embedded in the cleaned RI file.
returns = ri.pct_change(axis=1, fill_method=None)

print("Returns panel shape:", returns.shape)

# =========================================================
# PREPARE ANNUAL CO2 PANEL
# =========================================================
print("Preparing annual CO2 panel...")

co2_year_cols = list(pd.to_numeric(co2_raw.columns[2:], errors="raise").astype(int))
co2 = co2_raw.copy()
co2.columns = ["NAME", "ISIN"] + co2_year_cols
co2 = co2.set_index(["ISIN", "NAME"])
co2 = to_numeric_panel(co2)

# NOTE:
# We do NOT apply carry-forward here, because the cleaned annual CO2 file
# is assumed to already contain the cleaning choices you made.

print("CO2 panel shape:", co2.shape)

# Common index across RI and CO2
common_index = returns.index.intersection(co2.index)
ri = ri.loc[common_index]
returns = returns.loc[common_index]
co2 = co2.loc[common_index]

# =========================================================
# PREPARE RISK-FREE RATE
# =========================================================
print("Preparing risk-free rate...")

rf = rf_raw.copy()
if rf.shape[1] < 2:
    raise ValueError("RF.csv must contain at least one date column and one RF column.")

rf_date_col = rf.columns[0]
rf_value_col = rf.columns[1]

rf_dates = parse_rf_dates(rf[rf_date_col])
rf_values = pd.to_numeric(
    rf[rf_value_col].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

rf_series = pd.Series(rf_values.to_numpy(), index=rf_dates).sort_index()
rf_series = rf_series[~rf_series.index.isna()]
rf_series = rf_series.loc[rf_series.index <= pd.Timestamp("2025-12-31")]

# If RF looks like percentages > 1, scale down to decimals
rf_nonmissing = rf_series.dropna()
if len(rf_nonmissing) > 0 and rf_nonmissing.median() > 1:
    rf_series = rf_series / 100.0

print("RF series length:", len(rf_series))

# =========================================================
# BUILD YEAR-END MAP
# =========================================================
print("Building year-end map...")

december_dates = [d for d in ri.columns if d.month == 12]
year_end_dec = {}

for y in sorted({d.year for d in december_dates}):
    year_dates = [d for d in december_dates if d.year == y]
    if year_dates:
        year_end_dec[y] = max(year_dates)

required_years = range(START_FORMATION_YEAR, END_FORMATION_YEAR + 1)
missing_years = [y for y in required_years if y not in year_end_dec]
if missing_years:
    raise ValueError(f"Missing December RI dates for years: {missing_years}")

# =========================================================
# INVESTMENT-SET FUNCTION
# =========================================================
def get_investment_set(formation_year: int) -> pd.MultiIndex:
    dec_date = year_end_dec[formation_year]
    window_start = dec_date - pd.DateOffset(months=119)

    window_returns = returns.loc[
        :,
        (returns.columns >= window_start) & (returns.columns <= dec_date)
    ]

    # Rule 1: carbon available at end of year Y
    if formation_year in co2.columns:
        carbon_available = co2[formation_year].notna()
    else:
        carbon_available = pd.Series(False, index=co2.index)

    # Rule 2: valid December RI at end of year Y
    # (if price is missing at end of Y, do not invest in Y+1)
    valid_december_ri = ri[dec_date].notna()

    # Rule 3: enough monthly return history
    valid_return_counts = window_returns.notna().sum(axis=1)
    sufficient_history = valid_return_counts >= MIN_OBS_WINDOW

    # Rule 4: stale-price filter
    # Following the literal 10-year window interpretation: denominator = 120
    zero_counts = window_returns.eq(0).sum(axis=1)
    zero_prop = zero_counts / 120.0
    not_stale = zero_prop <= STALE_ZERO_THRESHOLD

    eligible_mask = carbon_available & valid_december_ri & sufficient_history & not_stale
    return common_index[eligible_mask.loc[common_index].to_numpy()]

# =========================================================
# MIN-VARIANCE SOLVER
# =========================================================
def solve_minvar(cov_matrix: pd.DataFrame) -> pd.Series:
    n = cov_matrix.shape[0]
    if n == 0:
        raise ValueError("Covariance matrix is empty.")
    if n == 1:
        return pd.Series([1.0], index=cov_matrix.index)

    Sigma = cov_matrix.to_numpy(dtype=float)
    Sigma = Sigma + RIDGE * np.eye(n)
    x0 = np.repeat(1.0 / n, n)

    def objective(w: np.ndarray) -> float:
        return float(w @ Sigma @ w)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * n

    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": SOLVER_MAXITER, "ftol": FTOL, "disp": False},
    )

    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")

    weights = pd.Series(res.x, index=cov_matrix.index)
    weights = weights / weights.sum()
    return weights

# =========================================================
# MAIN YEARLY LOOP
# =========================================================
print("Running yearly minimum-variance loop...")

investment_set_rows = []
weights_long_rows = []
oos_rows = []

for Y in range(START_FORMATION_YEAR, END_FORMATION_YEAR + 1):
    print(f"  Formation year {Y} -> OOS year {Y + 1}")

    eligible = get_investment_set(Y)
    dec_date = year_end_dec[Y]
    window_start = dec_date - pd.DateOffset(months=119)

    est_window = returns.loc[
        eligible,
        (returns.columns >= window_start) & (returns.columns <= dec_date)
    ]

    investment_set_rows.append({"formation_year": Y, "n_eligible": len(eligible)})

    if len(eligible) == 0:
        print(f"    WARNING: no eligible firms for year {Y}")
        continue

    # Practical covariance stability filter
    est_window = est_window.loc[est_window.notna().sum(axis=1) >= 2]
    if est_window.empty:
        print(f"    WARNING: no firms remain after covariance stability filter for year {Y}")
        continue

    # Fill missing estimation-window returns with each asset's own mean
    # (practical choice for covariance estimation)
    asset_means = est_window.mean(axis=1)
    est_window_filled = est_window.T.fillna(asset_means).T

    Sigma = est_window_filled.T.cov()

    # Drop any residual problematic rows/cols
    valid_assets = Sigma.index[~Sigma.isna().all(axis=1)]
    Sigma = Sigma.loc[valid_assets, valid_assets]
    Sigma = Sigma.dropna(axis=0, how="any").dropna(axis=1, how="any")

    if Sigma.empty:
        print(f"    WARNING: covariance matrix empty after cleanup for year {Y}")
        continue

    w0 = solve_minvar(Sigma)

    for (isin, name), w in w0.items():
        weights_long_rows.append({
            "formation_year": Y,
            "oos_year": Y + 1,
            "ISIN": isin,
            "NAME": name,
            "weight": w,
        })

    # Out-of-sample months in Y+1
    oos_months = sorted([d for d in returns.columns if d.year == Y + 1])
    if len(oos_months) == 0:
        continue

    w = w0.copy()

    for dt in oos_months:
        r_t = returns.loc[w.index, dt]

        # -------------------------------------------------
        # IMPORTANT:
        # This line is conservative and may need discussion.
        # It treats unexpected missing post-formation returns
        # as -100% losses.
        # If you do NOT want this convention, this is the main
        # line to revisit later.
        # -------------------------------------------------
        r_t = r_t.fillna(-1.0)

        rp_t = float((w * r_t).sum())

        oos_rows.append({
            "date": dt,
            "formation_year": Y,
            "oos_year": Y + 1,
            "portfolio_return": rp_t,
        })

        # Update weights passively within the year
        denom = 1.0 + rp_t
        if abs(denom) > 1e-12:
            w = w * (1.0 + r_t) / denom
            total = w.sum()
            if abs(total) > 1e-12:
                w = w / total
            else:
                w = pd.Series(np.repeat(1.0 / len(w), len(w)), index=w.index)
        else:
            w = pd.Series(np.repeat(1.0 / len(w), len(w)), index=w.index)

# =========================================================
# OUTPUT TABLES
# =========================================================
print("Saving outputs...")

investment_set_sizes = pd.DataFrame(investment_set_rows)
weights_by_year_long = pd.DataFrame(weights_long_rows)
minvar_monthly_returns = pd.DataFrame(oos_rows).sort_values("date").reset_index(drop=True)

if minvar_monthly_returns.empty:
    raise ValueError("No out-of-sample minimum-variance returns were produced.")

minvar_monthly_returns["date"] = pd.to_datetime(minvar_monthly_returns["date"])
minvar_monthly_returns = minvar_monthly_returns[
    (minvar_monthly_returns["date"] >= pd.Timestamp("2014-01-01")) &
    (minvar_monthly_returns["date"] <= pd.Timestamp("2025-12-31"))
].copy()

minvar_monthly_returns["cumulative_index"] = (1.0 + minvar_monthly_returns["portfolio_return"]).cumprod()
minvar_cumulative_returns = minvar_monthly_returns[
    ["date", "formation_year", "oos_year", "portfolio_return", "cumulative_index"]
].copy()

# Template-ready table: one row per month from 2014-01 to 2025-12
full_months = pd.date_range("2014-01-31", "2025-12-31", freq="ME")
template_ready = pd.DataFrame({"date": full_months})
template_ready = template_ready.merge(
    minvar_monthly_returns[["date", "portfolio_return"]],
    on="date",
    how="left"
)
template_ready["date_str"] = template_ready["date"].dt.strftime("%Y-%m")
template_ready = template_ready[["date_str", "portfolio_return"]]

# Summary stats
r = minvar_monthly_returns.set_index("date")["portfolio_return"].sort_index()
rf_monthly = rf_series.reindex(r.index)

if rf_monthly.isna().all():
    sharpe = np.nan
else:
    rf_monthly = rf_monthly.ffill().bfill()
    excess = r - rf_monthly
    ex_ann = annualized_return(excess)
    vol_ann = annualized_volatility(r)
    sharpe = ex_ann / vol_ann if pd.notna(ex_ann) and pd.notna(vol_ann) and vol_ann != 0 else np.nan

summary_stats = pd.DataFrame({
    "metric": [
        "Annualized average return",
        "Annualized volatility",
        "Sharpe ratio",
        "Minimum monthly return",
        "Maximum monthly return",
        "Final cumulative index",
        "Maximum drawdown",
        "Number of monthly observations",
    ],
    "value": [
        annualized_return(r),
        annualized_volatility(r),
        sharpe,
        r.min(),
        r.max(),
        minvar_monthly_returns["cumulative_index"].iloc[-1],
        max_drawdown(minvar_monthly_returns.set_index("date")["cumulative_index"]),
        int(r.notna().sum()),
    ]
})

# Save files
investment_set_sizes.to_csv(OUTPUT_DIR / "investment_set_sizes.csv", index=False)
weights_by_year_long.to_csv(OUTPUT_DIR / "minvar_weights_by_year_long.csv", index=False)
minvar_monthly_returns.to_csv(OUTPUT_DIR / "minvar_monthly_returns.csv", index=False)
minvar_cumulative_returns.to_csv(OUTPUT_DIR / "minvar_cumulative_returns.csv", index=False)
summary_stats.to_csv(OUTPUT_DIR / "minvar_summary_stats.csv", index=False)
template_ready.to_csv(OUTPUT_DIR / "minvar_template_ready.csv", index=False)

print("Done.")
print(f"Outputs saved in: {OUTPUT_DIR}")
print(investment_set_sizes.to_string(index=False))
print("\nSummary stats:")
print(summary_stats.to_string(index=False))

Loading cleaned CSV files...
Preparing monthly RI panel...
RI panel shape: (618, 313)
Constructing monthly simple returns...
Returns panel shape: (618, 313)
Preparing annual CO2 panel...
CO2 panel shape: (618, 27)
Preparing risk-free rate...
RF series length: 0
Building year-end map...
Running yearly minimum-variance loop...
  Formation year 2013 -> OOS year 2014
  Formation year 2014 -> OOS year 2015
  Formation year 2015 -> OOS year 2016
  Formation year 2016 -> OOS year 2017
  Formation year 2017 -> OOS year 2018
  Formation year 2018 -> OOS year 2019
  Formation year 2019 -> OOS year 2020
  Formation year 2020 -> OOS year 2021
  Formation year 2021 -> OOS year 2022
  Formation year 2022 -> OOS year 2023
  Formation year 2023 -> OOS year 2024
  Formation year 2024 -> OOS year 2025
Saving outputs...
Done.
Outputs saved in: ..\results\outputs_minvar
 formation_year  n_eligible
           2013         477
           2014         493
           2015         509
           2016         5

In [63]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()

# If the notebook is running from /notebooks, move one level up
PROJECT_DIR = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

# Try the most likely output locations
if (PROJECT_DIR / "outputs_minvar").exists():
    OUT_DIR = PROJECT_DIR / "outputs_minvar"
elif (PROJECT_DIR / "results" / "outputs_minvar").exists():
    OUT_DIR = PROJECT_DIR / "results" / "outputs_minvar"
else:
    raise FileNotFoundError(
        "Could not find 'outputs_minvar'. Checked both:\n"
        f"- {PROJECT_DIR / 'outputs_minvar'}\n"
        f"- {PROJECT_DIR / 'results' / 'outputs_minvar'}"
    )

# ----------------------------
# Load files
# ----------------------------
investment = pd.read_csv(OUT_DIR / "investment_set_sizes.csv")
monthly = pd.read_csv(OUT_DIR / "minvar_monthly_returns.csv")
cumulative = pd.read_csv(OUT_DIR / "minvar_cumulative_returns.csv")
weights = pd.read_csv(OUT_DIR / "minvar_weights_by_year_long.csv")
summary = pd.read_csv(OUT_DIR / "minvar_summary_stats.csv")

monthly["date"] = pd.to_datetime(monthly["date"])
cumulative["date"] = pd.to_datetime(cumulative["date"])

# ----------------------------
# Graph 1: Investment set size by formation year
# ----------------------------
plt.figure(figsize=(10, 6))
plt.plot(investment["formation_year"], investment["n_eligible"], marker="o")
plt.title("Eligible Universe Size by Formation Year")
plt.xlabel("Formation Year")
plt.ylabel("Number of Eligible Firms")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_investment_set_sizes.png", dpi=300)
plt.close()

# ----------------------------
# Graph 2: Cumulative performance
# ----------------------------
plt.figure(figsize=(12, 6))
plt.plot(cumulative["date"], cumulative["cumulative_index"])
plt.title("Minimum-Variance Portfolio Cumulative Performance")
plt.xlabel("Date")
plt.ylabel("Cumulative Index")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_cumulative_performance.png", dpi=300)
plt.close()

# ----------------------------
# Graph 3: Monthly returns through time
# ----------------------------
plt.figure(figsize=(12, 6))
plt.bar(monthly["date"], monthly["portfolio_return"])
plt.title("Minimum-Variance Portfolio Monthly Returns")
plt.xlabel("Date")
plt.ylabel("Monthly Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_monthly_returns.png", dpi=300)
plt.close()

# ----------------------------
# Graph 4: Annual returns bar chart
# ----------------------------
monthly["year"] = monthly["date"].dt.year
annual_returns = (
    monthly.groupby("year")["portfolio_return"]
    .apply(lambda x: (1 + x).prod() - 1)
    .reset_index()
)

plt.figure(figsize=(10, 6))
plt.bar(annual_returns["year"].astype(str), annual_returns["portfolio_return"])
plt.title("Minimum-Variance Portfolio Annual Returns")
plt.xlabel("Year")
plt.ylabel("Annual Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_annual_returns.png", dpi=300)
plt.close()

# ----------------------------
# Graph 5: Top 10 weights in latest formation year
# ----------------------------
latest_year = int(weights["formation_year"].max())
top_weights = (
    weights[weights["formation_year"] == latest_year]
    .sort_values("weight", ascending=False)
    .head(10)
    .sort_values("weight", ascending=True)
)

plt.figure(figsize=(10, 6))
plt.barh(top_weights["NAME"], top_weights["weight"])
plt.title(f"Top 10 Portfolio Weights - Formation Year {latest_year}")
plt.xlabel("Weight")
plt.ylabel("Company")
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_top10_weights_latest_year.png", dpi=300)
plt.close()

# ----------------------------
# Graph 6: Rolling 12-month return
# ----------------------------
rolling_12m = monthly[["date", "portfolio_return"]].copy()
rolling_12m["rolling_12m_return"] = (
    (1 + rolling_12m["portfolio_return"])
    .rolling(12)
    .apply(lambda x: x.prod() - 1, raw=True)
)

plt.figure(figsize=(12, 6))
plt.plot(rolling_12m["date"], rolling_12m["rolling_12m_return"])
plt.title("Rolling 12-Month Return")
plt.xlabel("Date")
plt.ylabel("Rolling 12-Month Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_rolling_12m_return.png", dpi=300)
plt.close()

print("Graphs created successfully.")
print()
print("Files created:")
print("- graph_investment_set_sizes.png")
print("- graph_cumulative_performance.png")
print("- graph_monthly_returns.png")
print("- graph_annual_returns.png")
print("- graph_top10_weights_latest_year.png")
print("- graph_rolling_12m_return.png")

Graphs created successfully.

Files created:
- graph_investment_set_sizes.png
- graph_cumulative_performance.png
- graph_monthly_returns.png
- graph_annual_returns.png
- graph_top10_weights_latest_year.png
- graph_rolling_12m_return.png
